# BÁO CÁO DỰ ÁN: Question Difficulty and Student Ability Discrimination 
## Học phần: Khoa Học Dữ Liệu

### 1.Giới thiệu mục tiêu
Dự án tập trung vào việc phân tích dữ liệu tương tác từ nền tảng học tập **Riiid**.
Mục tiêu chính là ứng dụng mô hình toán học để xác định độ khó của từng câu hỏi, từ đó đưa ra các đánh giá khách quan về chất lượng bộ đề thi.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
# đọc dữ liệu
train = pd.read_csv('sample_train.csv')
questions = pd.read_csv('questions.csv')
lectures = pd.read_csv('lectures.csv')

print("Đã đọc thành công 3 file dữ liệu.")

### 2. Mô tả dữ liệu (Data Description)
Dữ liệu bao gồm 3 file CSV. Trong đó:
- **sample_train.csv**: tương tác của học sinh (100.000 dòng).
  + timestamp: Thời gian (mili giây) từ lúc học sinh bắt đầu tương tác lần đầu.
  + <span style="color:red">user_id</span>: Mã định danh của học sinh.
  + <span style="color:red">content_id</span>: Mã định danh của câu hỏi hoặc bài giảng.
  + content_type_id: 0 nếu là câu hỏi, 1 nếu là bài giảng.
  + <span style="color:red">answered_correctly</span>: Kết quả: 1 là đúng, 0 là sai.
- **questions.csv**: thông tin chi tiết về các câu hỏi (id).
  + question_id: tương tự content_id trong file train.
  + bundle_id: Mã nhóm các câu hỏi dùng chung một đoạn văn/đoạn hội thoại.
  + correct_answer: Đáp án đúng của câu hỏi đó.
  + part: Phần thi
  + tags: Các mã kỹ năng
- **lectures.csv**: Thông tin các bài giảng (id, tag, part, type)
  + lecture_id: Mã bài giảng 
  + part: Bài giảng đó thuộc phần kiến thức nào.
  + tag: Chủ đề của bài giảng.
  + type_of: Loại bài giảng

### 3. Tiền xử lý dữ liệu
Để đảm bảo tính chính xác cho mô hình IRT, thực hiện:
- Loại bỏ các dòng tương tác là bài giảng (`content_type_id == 1`)=> chỉ giữ lại các dòng là câu hỏi.
- Giữ lại các dòng làm bài trắc nghiệm có kết quả hợp lệ ( `answered_correctly= 0 or answered_correctly= 1` ).

In [ ]:
# Lọc dữ liệu: chỉ lấy câu hỏi, tạo bản sao của train.
df_clean = train[train['content_type_id'] == 0].copy()
df_clean = train_clean[(train_clean['answered_correctly']) == 0 | (df_clean['answered_correctly'] == 1)]

print(f"Kích thước dữ liệu sau khi làm sạch: {df_clean.shape}")
print(f"Số lượng học sinh trong mẫu: {df_clean['user_id'].nunique()}")

In [ ]:
# Biểu đồ tỷ lệ Đúng/Sai 
plt.figure(figsize=(7, 5)) #plt.figure: Khởi tạo một khung hình để vẽ với chiều ngang 7 đơn vị chiều dọc 5 đơn vị
sns.countplot(x = 'answered_correctly', data = df_clean, hue = 'answered_correctly') #hue='answered_correctly': Tô màu khác nhau dựa trên giá trị (đúng/sai).
plt.title('Tỷ lệ câu trả lời Đúng (1) và Sai (0)')
plt.show()

# Tính tỷ lệ đúng trung bình
mean_correct = df_clean['answered_correctly'].sum() / len(df_clean)
print(f"Tỷ lệ trả lời đúng trung bình của mẫu: {mean_correct:.2%}")


**Nhận xét:** Dựa vào biểu đồ, ta thấy tỷ lệ trả lời đúng chiếm khoảng 65.60%. Điều này cho thấy tập câu hỏi mẫu có độ khó vừa phải, phù hợp để triển khai các bước tính toán tiếp theo.